In [ ]:
#| default_exp handlers.general

# General Handler

`encode(yaml_path)` — marisco 2.0 のグランドマスター関数。
YAML パス 1 つを渡すだけで、全 3 フェーズ（loader → transformer → writer）を直線的に駆動し、
生 CSV/TSV を MARIS 正典 NetCDF4 へ自動整流・製本する。

In [ ]:
#| export
from __future__ import annotations
import importlib
from pathlib import Path
from marisco.callbacks import Transformer
from marisco.handlers.pipeline.loader  import HandlerConfig, PluginSpec, load_data, gap_check
from marisco.handlers.pipeline.writer  import write_netcdf

## encode

| フェーズ | 呼び出し | 役割 |
|--------|---------|------|
| Phase 0 | `HandlerConfig.from_yaml` + `gap_check` | 型安全バインド ＆ 必須列の事前検疫 |
| Phase 1 | `load_data` | 生 CSV/TSV → `{grp: DataFrame}` |
| Phase 2 | `_load_plugin(pre_cbs)` + `build_chain` + `_load_plugin(post_cbs)` | プラグイン前段 + 11本 Soft CB + プラグイン後段（if文ゼロ） |
| Phase 3 | `write_netcdf` | 2フェーズ Strict ガード → `.nc` 物理書き出し |

`pre_cbs` / `post_cbs` が空リスト（YAML 省略時のデフォルト）の場合、アンパック展開で透過 no-op。

In [ ]:
#| export
def _load_plugin(spec: PluginSpec):
    "Fail-Fast dynamic import: ImportError/AttributeError raised before pipeline starts."
    module_path, class_name = spec.path.rsplit(".", 1)
    cls = getattr(importlib.import_module(module_path), class_name)
    return cls(**spec.args)

def _load_plugin_fn(spec: PluginSpec):
    "Fail-Fast dynamic import of a plain function (not instantiated); for custom loaders."
    module_path, fn_name = spec.path.rsplit(".", 1)
    return getattr(importlib.import_module(module_path), fn_name)

def build_core_pipeline(cfg: HandlerConfig) -> list:
    "Auto-assemble the standard core CB chain with topology guards; EncodeTimeCB/SanitizeLonLatCB degrade to Null-Object when their required columns are absent."
    from marisco.callbacks import (
        RenameColsCB, SoftParseDateTimeCB, SoftMeltWideNuclidesCB,
        SoftConvertUnitCB, SoftRemapCB, EncodeTimeCB, SanitizeLonLatCB, AddSampleIDCB,
        _GuardedEncodeTimeCB, _GuardedSanitizeLonLatCB,
    )


    # Merge columns shorthand (S-7c) with legacy rename; time_format hint wins over dt_format
    merged = cfg.model_copy(update={
        "rename":    {**cfg.columns, **cfg.rename},
        "dt_format": cfg.time_format or cfg.dt_format,
    })
    col_provider = next((v for v in merged.rename.values() if v.endswith('_PROVIDER')), None)

    return [
        RenameColsCB(mapping=merged.rename, string_cast=merged.string_cast),
        SoftParseDateTimeCB(col_date=merged.col_date, col_time=merged.col_time, fmt=merged.dt_format),
        SoftMeltWideNuclidesCB(spec=[s.model_dump() for s in merged.melt_spec]),
        *[SoftConvertUnitCB(rule=r.model_dump()) for r in merged.unit_conversions],
        SoftRemapCB(col_src='NUCLIDE', col_remap='NUCLIDE', lut=merged.nuclide_lut),
        SoftRemapCB(col_src='UNIT',    col_remap='UNIT',    lut=merged.unit_lut),
        SoftRemapCB(col_src='LAB',     col_remap='LAB',     lut=merged.lab_lut),
        SoftRemapCB(col_src='NUCLIDE', col_remap='AREA',    lut={}, default_val=merged.area_default),
        _GuardedSanitizeLonLatCB(),
        _GuardedEncodeTimeCB(),
        AddSampleIDCB(col_provider=col_provider),
    ]

def encode(yaml_path: str | Path, fname_out: str = None) -> None:
    "Encode any YAML-configured dataset to MARIS NetCDF4 in a pure, single-pass pipeline."
    from marisco.callbacks import LowerStripNameCB
    cfg = HandlerConfig.from_yaml(yaml_path)
    gap_check(cfg)
    if fname_out:
        cfg = cfg.model_copy(update={"fname_out": fname_out})
    loader  = _load_plugin_fn(cfg.loader) if cfg.loader else load_data
    dfs     = loader(cfg)
    # normalize_case CBs run FIRST (before pre_cbs) to satisfy topology: e.g. nuclide→NUCLIDE
    # must exist before any pre_cb that reads NUCLIDE (such as HelcomNuclideRemapCB).
    normalize_cbs = [LowerStripNameCB(col_src=s, col_dst=d) for s, d in cfg.normalize_case.items()]
    chain   = [*normalize_cbs,
               *[_load_plugin(s) for s in cfg.pre_cbs],
               *build_core_pipeline(cfg),
               *[_load_plugin(s) for s in cfg.post_cbs]]
    tfm     = Transformer(dfs, cbs=chain)
    tfm()
    write_netcdf(tfm, cfg)

In [ ]:
import sys
sys.stdout.reconfigure(encoding="utf-8")
from marisco.handlers.pipeline.loader import HandlerConfig, gap_check
from marisco.handlers.general import encode, _load_plugin, build_core_pipeline

cfg = HandlerConfig.from_yaml("config/handlers/fram_strait.yaml")
gap_check(cfg)

core = build_core_pipeline(cfg)
print(f"build_core_pipeline (FramStrait, 1 unit_conv): {len(core)} CBs")
for i, cb in enumerate(core, 1):
    print(f"  {i:2d}. {type(cb).__name__}")

# Topology guard smoke test: empty dfs → guards degrade to Null-Object
from marisco.callbacks import Transformer
import pandas as pd
tfm_empty = Transformer({'SEAWATER': pd.DataFrame()}, cbs=core)
tfm_empty()  # must not raise KeyError for TIME/LON/LAT absent
print("\nTopology guard ✓ — no KeyError on empty DataFrame (Null-Object auto-degrade)")

# normalize_case smoke test
cfg_norm = cfg.model_copy(update={"normalize_case": {"nuclide_raw": "NUCLIDE"}})
from marisco.callbacks import LowerStripNameCB
norm_cbs = [LowerStripNameCB(col_src=s, col_dst=d) for s, d in cfg_norm.normalize_case.items()]
assert len(norm_cbs) == 1 and type(norm_cbs[0]).__name__ == "LowerStripNameCB"
print("normalize_case → LowerStripNameCB ✓")

# Fail-Fast test: bad path raises immediately
try:
    _load_plugin(type("S", (), {"path": "marisco.nonexistent.FakeCB", "args": {}})())
except (ImportError, ModuleNotFoundError) as e:
    print(f"Fail-Fast ✓ — bad plugin path raises: {type(e).__name__}")

In [ ]:
#|eval: false
# Full pipeline integration test (requires network access to Zenodo)
encode("config/handlers/fram_strait.yaml", fname_out="_data/output/FramStrait_general.nc")
print("FramStrait_general.nc written.")